In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

In [0]:
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

SCHEMA = "skypoints"

BRONZE_TABLE = f"{catalog}.{SCHEMA}.bronze_member_feed"

SILVER_TABLE = f"{catalog}.{SCHEMA}.silver_member"

REJECT_TABLE = f"{catalog}.{SCHEMA}.silver_member_rejects"

print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)
print("Reject:", REJECT_TABLE)


In [0]:
bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("SOURCE_COUNTRY").isin("IND", "USA", "AUS"))
)

print("Bronze records:", bronze_df.count())

display(bronze_df)

Extract original JSON records from the bronze table

In [0]:
map_schema = T.MapType(
    T.StringType(),
    T.StringType(),
    True
)

parsed_df = bronze_df.withColumn(
    "RECORD_DATA",
    F.from_json(F.col("RAW_RECORD"), map_schema)
)

In [0]:
from pyspark.sql import functions as F

def get_source_column(field_mapping):
    result = F.lit(None).cast("string")

    for source_country, source_column in field_mapping.items():
        result = F.when(
            F.col("SOURCE_COUNTRY") == source_country,
            F.col("RECORD_DATA").getItem(source_column)
        ).otherwise(result)

    return result

In [0]:
parsed_df= parsed_df.withColumn(
    "MEMBER_ID",
    get_source_column({
        "IND": "ID",
        "USA": "ID",
        "AUS": "Unique ID"
    })
)

standardize all the fields

In [0]:
field_mappings = {
    "MEMBER_NAME": {
        "IND": "Name",
        "USA": "Name",
        "AUS": "Member Name"
    },

    "DOB_RAW": {
        "IND": "DOB",
        "USA": "DOB",
        "AUS": "Date of Birth"
    },

    "TIER_CODE": {
        "IND": "TierCode",
        "USA": "TierCode",
        "AUS": "Tier Type"
    },

    "ENROLLMENT_DATE_RAW": {
        "IND": "EnrollmentDate",
        "USA": "EnrollmentDate",
        "AUS": "Date of Enrollment"
    },

    "FLIGHT_DATE_RAW": {
        "IND": "Flight Date",
        "USA": "FlightDate",
        "AUS": "Date of Flight"
    },

    "LAST_UPDATED_RAW": {
        "IND": "Last_Updated_Date",
        "USA": "Last_Updated_Date",
        "AUS": "Last_Updated_Date"
    },

    "MEMBER_TYPE": {
        "IND": "Individual or Corporate"
    }
}

In [0]:
for target_column, mapping in field_mappings.items():

    parsed_df = parsed_df.withColumn(
        target_column,
        get_source_column(mapping)
    )

In [0]:
display(parsed_df)

Standardize dates

In [0]:
# Remove leading/trailing spaces
date_columns = [
    "DOB_RAW",
    "ENROLLMENT_DATE_RAW",
    "FLIGHT_DATE_RAW"
]

for column in date_columns:
    parsed_df = parsed_df.withColumn(
        column,
        F.trim(F.col(column))
    )

In [0]:
display(parsed_df.select(date_columns))

In [0]:
from pyspark.sql import functions as F

def safe_to_date(raw_column, date_format):
    return F.to_date(
        F.try_to_timestamp(
            F.col(raw_column),
            F.lit(date_format)
        )
    )


def standardize_date(column_name):

    raw = F.trim(F.col(column_name))
    country = F.col("SOURCE_COUNTRY")

    usa_digits = F.regexp_replace(raw, r"[^0-9]", "")
    usa_normalized = F.lpad(usa_digits, 8, "0")

    return (
        F.when(
            country == "IND",
            safe_to_date(column_name, "M/d/yyyy")
        )
        .when(
            country == "USA",
            F.to_date(
                F.try_to_timestamp(usa_normalized, F.lit("MMddyyyy"))
            )
        )
        .when(
            country == "AUS",
            safe_to_date(column_name, "yyyy-MM-dd")
        )
        .otherwise(F.lit(None).cast("date"))
    )

In [0]:
parsed_df = (
    parsed_df
    .withColumn("DOB", standardize_date("DOB_RAW"))
    .withColumn("ENROLLMENT_DATE", standardize_date("ENROLLMENT_DATE_RAW"))
    .withColumn("FLIGHT_DATE", standardize_date("FLIGHT_DATE_RAW"))
)

In [0]:
parsed_df = parsed_df.withColumn(
    "LAST_UPDATED_DATE",
    F.to_timestamp(
        F.trim(F.col("LAST_UPDATED_RAW")),
        "yyyy-MM-dd HH:mm:ss"
    )
)

In [0]:
display(
    parsed_df.select(
        "SOURCE_COUNTRY",
        "MEMBER_ID",
        "MEMBER_NAME",
        "DOB_RAW",
        "DOB",
        "ENROLLMENT_DATE_RAW",
        "ENROLLMENT_DATE",
        "FLIGHT_DATE_RAW",
        "FLIGHT_DATE",
        "LAST_UPDATED_DATE"
    )
)

Calculate Age and Stale_Member

In [0]:
# Age

parsed_df = parsed_df.withColumn(
    "AGE",
    F.when(
        F.col("DOB").isNotNull(),
        F.floor(
            F.months_between(F.current_date(),F.col("DOB"))
        ) / 12
    ).cast("integer")
)

In [0]:
# Stale Member

parsed_df = parsed_df.withColumn(
    "STALE_MEMBER",
    F.when(
        F.col("FLIGHT_DATE").isNull(), F.lit("UNKNOWN")
    )
    .when(
        F.datediff(F.current_date(), F.col("FLIGHT_DATE")) > 90, F.lit("Y")
    ).otherwise(F.lit("N"))
)

Implement data-quality validations

In [0]:
parsed_df = parsed_df.withColumn(
    "DQ_ERRORS",
    F.concat_ws(
        " | ",
        F.when(
            F.col("MEMBER_ID").isNull() | 
            (F.trim(F.col("MEMBER_ID")) == ""),
            "MISSING_MEMBER_ID"
        ),
        F.when(
            F.col("MEMBER_NAME").isNull() |
            (F.trim(F.col("MEMBER_NAME")) == ""),
            "MISSING_MEMBER_NAME"
        ),
        F.when(
            F.col("ENROLLMENT_DATE_RAW").isNull() |
            (F.col("ENROLLMENT_DATE_RAW") == ""),
            "MISSING_ENROLLMENT_DATE"
        ),
         F.when(
            (F.col("ENROLLMENT_DATE_RAW").isNotNull()) &
            (F.col("ENROLLMENT_DATE_RAW") != "") &
            F.col("ENROLLMENT_DATE").isNull(),
            "INVALID_ENROLLMENT_DATE"
        ),
         F.when(
            F.col("DOB_RAW").isNotNull() &
            (F.col("DOB_RAW") != "") &
            F.col("DOB").isNull(),
            "INVALID_DOB"
        ),
         F.when(
            F.col("DOB") > F.current_date(),
            "DOB_IN_FUTURE"
        ),
         F.when(
            F.col("FLIGHT_DATE_RAW").isNotNull() &
            (F.col("FLIGHT_DATE_RAW") != "") &
            F.col("FLIGHT_DATE").isNull(),
            "INVALID_FLIGHT_DATE"
        ),
         F.when(
            F.col("FLIGHT_DATE") > F.current_date(),
            "FLIGHT_DATE_IN_FUTURE"
        ),

        F.when(
            F.col("LAST_UPDATED_DATE").isNull(),
            "INVALID_LAST_UPDATED_DATE"
        ),

        F.when(
            F.col("ENROLLMENT_DATE") >
            F.to_date(F.col("LAST_UPDATED_DATE")),
            "ENROLLMENT_AFTER_LAST_UPDATE"
        )

    )
)

In [0]:
parsed_df = parsed_df.withColumn(
    "MEMBER_ID",
    F.trim(F.col("MEMBER_ID"))
).withColumn(
    "MEMBER_NAME",
    F.trim(F.col("MEMBER_NAME"))
)

duplicate_window = Window.partitionBy(
    "SOURCE_COUNTRY",
    "MEMBER_ID"
)

parsed_df = parsed_df.withColumn(
    "ID_COUNT",
    F.count("*").over(duplicate_window)
)

In [0]:
# duplicate member id data quality check

parsed_df = parsed_df.withColumn(
    "DQ_ERRORS",
    F.concat_ws(
        " | ",
        F.col("DQ_ERRORS"),
        F.when(
            F.col("ID_COUNT") > 1,
            "DUPLICATE_MEMBER_ID_WITHIN_COUNTRY"
        )
    )
)

In [0]:
final_df = parsed_df.withColumn(
    "RECORD_STATUS",
    F.when(
        F.col("DQ_ERRORS") == "", "VALID"
    ).otherwise("REJECTED")
)

In [0]:
silver_valid_df = (
    final_df.filter(F.col("RECORD_STATUS") == "VALID")
    .select(
       "MEMBER_ID",
        "MEMBER_NAME",
        "DOB",
        "AGE",
        "TIER_CODE",
        "ENROLLMENT_DATE",
        "FLIGHT_DATE",
        "STALE_MEMBER",
        "MEMBER_TYPE",
        "SOURCE_COUNTRY",
        "SOURCE_FILE_NAME",
        "LAST_UPDATED_DATE",
        "BATCH_ID",
        "RECORD_HASH" 
    )
)

In [0]:
silver_reject_df = (
    final_df
    .filter(F.col("RECORD_STATUS") == "REJECTED")
    .select(
        "MEMBER_ID",
        "MEMBER_NAME",
        "SOURCE_COUNTRY",
        "SOURCE_FILE_NAME",
        "DOB_RAW",
        "ENROLLMENT_DATE_RAW",
        "FLIGHT_DATE_RAW",
        "DQ_ERRORS",
        "BATCH_ID",
        "RECORD_HASH"
    )
)

Write Silver Delta tables

In [0]:
(
    silver_valid_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

(
    silver_reject_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REJECT_TABLE)
)

Validate Silver results

In [0]:
%sql
-- Valid versus rejected records
SELECT
    RECORD_STATUS,
    COUNT(*) AS RECORD_COUNT
FROM (
    SELECT 'VALID' AS RECORD_STATUS
    FROM IDENTIFIER('skypoints.silver_member')

    UNION ALL

    SELECT 'REJECTED' AS RECORD_STATUS
    FROM IDENTIFIER('skypoints.silver_member_rejects')
)
GROUP BY RECORD_STATUS;

In [0]:
%sql
-- Inspect standardized members
SELECT
    MEMBER_ID,
    MEMBER_NAME,
    SOURCE_COUNTRY,
    DOB,
    AGE,
    ENROLLMENT_DATE,
    FLIGHT_DATE,
    STALE_MEMBER
FROM skypoints.silver_member
ORDER BY SOURCE_COUNTRY, MEMBER_ID;

In [0]:
%sql
-- Inspect rejected records
SELECT
    MEMBER_ID,
    MEMBER_NAME,
    SOURCE_COUNTRY,
    DQ_ERRORS
FROM skypoints.silver_member_rejects;

In [0]:
%sql
--Verify stale-member calculation
SELECT
    MEMBER_ID,
    MEMBER_NAME,
    FLIGHT_DATE,
    DATEDIFF(CURRENT_DATE(), FLIGHT_DATE) AS DAYS_SINCE_FLIGHT,
    STALE_MEMBER
FROM skypoints.silver_member;

In [0]:
%sql
--Check duplicate Member IDs
SELECT
    SOURCE_COUNTRY,
    MEMBER_ID,
    COUNT(*) AS RECORD_COUNT
FROM skypoints.silver_member
GROUP BY SOURCE_COUNTRY, MEMBER_ID
HAVING COUNT(*) > 1;